# 🔑 Solutions: Lesson 14 · Deep Learning Optimisation & Regularisation

> **Instructor's note.** Don't read these until you've really tried! For each problem: read the statement, look at the solution, then *close the notebook and re-implement it from memory* in the lesson notebook. The self-checks below are identical to the ones you saw there.

Each solution cell is self-contained and runs top-to-bottom.

In [1]:
import sys; sys.path.append("..")
import numpy as np
import matplotlib.pyplot as plt
from utils import dlcourse as dl
import pandas as pd, math
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import TensorDataset, DataLoader
    cpu = torch.device("cpu")
except ImportError:
    pass
dl.set_seed(42)
%matplotlib inline

---
### 🧩 Problem 14.1: Inverted dropout  &nbsp; `⭐⭐`

Implement `dropout(x, p, training, rng)` (NumPy). In training mode: zero each element with probability `p` and scale the kept ones by `1/(1−p)`. In eval mode: return `x` unchanged. Use the provided `rng` (`np.random.default_rng`) for the random mask.


In [2]:
def dropout(x, p, training, rng):
    if not training or p == 0:
        return x
    keep = rng.random(x.shape) >= p
    return x * keep / (1 - p)

In [3]:
# ▶ self-check
rng = np.random.default_rng(0); x = np.ones(200000)
y = dropout(x, 0.3, True, rng)
dl.check("about 30% zeroed", y is not None and abs((y == 0).mean() - 0.3) < 0.01)
dl.check("survivors are scaled to 1/(1-p)", y is not None and np.allclose(y[y != 0], 1 / 0.7))
dl.check("expected value preserved (mean ≈ 1)", y is not None and abs(y.mean() - 1) < 0.01)
dl.check("eval mode is the identity", np.array_equal(dropout(x, 0.3, False, rng), x))

✅ about 30% zeroed
✅ survivors are scaled to 1/(1-p)
✅ expected value preserved (mean ≈ 1)
✅ eval mode is the identity


**Why this works.** Scaling the survivors by 1/(1−p) keeps the *expected* activation unchanged, so no adjustment is needed at test time.

---
### 🧩 Problem 14.2: Batch-norm forward pass  &nbsp; `⭐⭐`

Implement `batchnorm_forward(x, gamma, beta, eps=1e-5)` for `x` of shape `(batch, features)` using **batch** statistics (biased variance, `unbiased=False`/`ddof=0`). The check compares with `nn.BatchNorm1d` in training mode.


In [4]:
def batchnorm_forward(x, gamma, beta, eps=1e-5):
    mu, var = x.mean(0), x.var(0, unbiased=False)
    return gamma * (x - mu) / torch.sqrt(var + eps) + beta

In [5]:
# ▶ self-check
torch.manual_seed(1); x = torch.randn(16, 5) * 3 + 2
bn = nn.BatchNorm1d(5); nn.init.uniform_(bn.weight, 0.5, 1.5); nn.init.uniform_(bn.bias, -1, 1); bn.train()
out = batchnorm_forward(x, bn.weight, bn.bias)
dl.check("matches nn.BatchNorm1d (train mode)", out is not None and torch.allclose(out, bn(x), atol=1e-5))
z = batchnorm_forward(x, torch.ones(5), torch.zeros(5))
dl.check("with γ=1, β=0 each feature has mean 0 and std 1", z is not None and torch.allclose(z.mean(0), torch.zeros(5), atol=1e-5) and torch.allclose(z.std(0, unbiased=False), torch.ones(5), atol=1e-3))

✅ matches nn.BatchNorm1d (train mode)
✅ with γ=1, β=0 each feature has mean 0 and std 1


---
### 🧩 Problem 14.3: Cosine schedule by formula  &nbsp; `⭐⭐`

Implement `cosine_lr(t, T, lr_max, lr_min=0.0)` = `lr_min + ½(lr_max − lr_min)(1 + cos(π·t/T))` for step `t` out of `T`. Compare with PyTorch's `CosineAnnealingLR`.


In [6]:
def cosine_lr(t, T, lr_max, lr_min=0.0):
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + math.cos(math.pi * t / T))

In [7]:
# ▶ self-check
dl.check("starts at lr_max", cosine_lr(0, 50, 0.1) is not None and np.isclose(cosine_lr(0, 50, 0.1), 0.1))
dl.check("midpoint is lr_max/2", cosine_lr(25, 50, 0.1) is not None and np.isclose(cosine_lr(25, 50, 0.1), 0.05))
p = nn.Parameter(torch.zeros(1)); opt = torch.optim.SGD([p], lr=0.1); sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, 50, eta_min=0.001)
ok = True
for t in range(50):
    v = cosine_lr(t, 50, 0.1, 0.001)
    ok &= v is not None and np.isclose(v, opt.param_groups[0]["lr"], atol=1e-9)
    opt.step(); sch.step()
dl.check("matches torch's CosineAnnealingLR at every step", ok)

✅ starts at lr_max
✅ midpoint is lr_max/2


✅ matches torch's CosineAnnealingLR at every step


---
### 🧩 Problem 14.4: Early-stopping logic  &nbsp; `⭐⭐`

Write `early_stopping(val_losses, patience)`. Walk through the validation losses epoch by epoch (epochs numbered from 1). Track the best loss so far and the number of epochs since it last improved (strictly lower).
Stop as soon as that counter reaches `patience`. Return `(stop_epoch, best_epoch)`; if training never triggers a stop, `stop_epoch` is the last epoch.


In [8]:
def early_stopping(val_losses, patience):
    best, best_epoch, wait = float("inf"), 0, 0
    for epoch, loss in enumerate(val_losses, 1):
        if loss < best:
            best, best_epoch, wait = loss, epoch, 0
        else:
            wait += 1
        if wait >= patience:
            return epoch, best_epoch
    return len(val_losses), best_epoch

In [9]:
# ▶ self-check
losses = [1.0, 0.8, 0.6, 0.55, 0.58, 0.57, 0.56, 0.54, 0.6]
dl.check("patience 3 → stops at epoch 7, best was epoch 4", early_stopping(losses, 3) == (7, 4))
dl.check("patience 5 → epoch 8 improves (0.54), so no stop until the end: (9, 8)", early_stopping(losses, 5) == (9, 8))
dl.check("always improving → never stops early", early_stopping([5, 4, 3, 2, 1], 2) == (5, 5))

✅ patience 3 → stops at epoch 7, best was epoch 4
✅ patience 5 → epoch 8 improves (0.54), so no stop until the end: (9, 8)
✅ always improving → never stops early


---
### 🧩 Problem 14.5: L2 penalty = weight decay  &nbsp; `⭐`

Write `sgd_step_l2(w, grad, lr, lam)` returning the new weights after a plain SGD step on `loss + (lam/2)·Σw²`: i.e. `w − lr·(grad + lam·w)`. Then confirm it equals PyTorch's `SGD(weight_decay=lam)` (which does exactly this).


In [10]:
def sgd_step_l2(w, grad, lr, lam):
    return w - lr * (grad + lam * w)

In [11]:
# ▶ self-check
w0 = torch.tensor([1.0, -2.0, 3.0]); g = torch.tensor([0.5, 0.5, -1.0])
mine = sgd_step_l2(w0.clone(), g, 0.1, 0.01)
p = nn.Parameter(w0.clone()); opt = torch.optim.SGD([p], lr=0.1, weight_decay=0.01); p.grad = g.clone(); opt.step()
dl.check("matches torch SGD(weight_decay)", mine is not None and torch.allclose(mine, p.detach()))
dl.check("weights shrink even with zero data gradient", torch.all(sgd_step_l2(w0.clone(), torch.zeros(3), 0.1, 0.1).abs() < w0.abs()))

✅ matches torch SGD(weight_decay)
✅ weights shrink even with zero data gradient
